# Accompanying Code for Chronobiological Features for Long-Term CGM Dysregulation

Reproduces the machine-learning figures and results from Burks et al., *PLOS Digital Health* (2025),
[doi:10.1371/journal.pdig.0000815](https://doi.org/10.1371/journal.pdig.0000815).

Reusable logic lives in the `src/` package; the cells below orchestrate it. Download
`processed_CGM_data_for_ML.parquet` from [doi:10.6075/J0BR8SK9](https://doi.org/10.6075/J0BR8SK9)
and place it in the repository root before running.

## Setup

In [ ]:
!pip install -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from src import (
    load_dataset, split_features_label, stratified_split, feature_matrix,
    build_classifier, cross_validate_report, per_class_roc_auc,
    plot_calibration_curves, plot_feature_importance_comparison, pca_biplot,
)

import warnings
warnings.filterwarnings('ignore')

# Single random seed for every split/model in the analysis (reproducibility).
RANDOM_STATE = 42

## Model 1: Statistical Features Only

Whole-day statistical summaries only (no complexity, temporal, or intraday sub-window features).

In [ ]:
# Load, reduce to Model 1's feature set, and make an 80/20 split
# stratified jointly on PID, Age, Gender, and Treatment.
data_stat = load_dataset(statistical_only=True)
X_stat, y_stat = split_features_label(data_stat)
Xtrain_s, Xtest_s, ytrain_s, ytest_s = stratified_split(X_stat, y_stat, random_state=RANDOM_STATE)
Xtrain_s.shape, Xtest_s.shape

### 5-fold cross-validation, test evaluation, and per-class ROC/AUC

In [ ]:
bst_stat = build_classifier(random_state=RANDOM_STATE)
metrics_stat = cross_validate_report(bst_stat, Xtrain_s, ytrain_s, Xtest_s, ytest_s,
                                     random_state=RANDOM_STATE)

# One-vs-rest AUC for each class (0 = lower, 1 = similar, 2 = greater area-above-range).
roc_stat = per_class_roc_auc(ytest_s, metrics_stat['proba'])
{c: round(roc_stat[c]['auc'], 3) for c in roc_stat}

### Calibration curves

In [ ]:
plot_calibration_curves(ytest_s, metrics_stat['proba'], title='Model 1 Calibration Curves')
plt.show()

## Model 2: Statistical + Temporal + Complexity Features

The full feature set, including the chronobiologically-informed complexity and temporal features.

In [ ]:
data_all = load_dataset(statistical_only=False)
X_all, y_all = split_features_label(data_all)
Xtrain_a, Xtest_a, ytrain_a, ytest_a = stratified_split(X_all, y_all, random_state=RANDOM_STATE)
Xtrain_a.shape, Xtest_a.shape

_Optional: hyperparameter grid search (the defaults in `build_classifier` are the selected best)._

In [ ]:
# for n_estimators in [50, 100, 200, 500]:
#     for max_depth in [2, 3, 4, 5, 6]:
#         model = build_classifier(random_state=RANDOM_STATE,
#                                  n_estimators=n_estimators, max_depth=max_depth)
#         from src import fit_predict
#         _, proba = fit_predict(model, Xtrain_a, ytrain_a, Xtest_a)
#         roc = per_class_roc_auc(ytest_a, proba)
#         print(n_estimators, max_depth, [round(roc[c]['auc'], 3) for c in roc])

### 5-fold cross-validation, test evaluation, and per-class ROC/AUC

In [ ]:
bst_all = build_classifier(random_state=RANDOM_STATE)
metrics_all = cross_validate_report(bst_all, Xtrain_a, ytrain_a, Xtest_a, ytest_a,
                                    random_state=RANDOM_STATE)

roc_all = per_class_roc_auc(ytest_a, metrics_all['proba'])
{c: round(roc_all[c]['auc'], 3) for c in roc_all}

### Calibration curves

In [ ]:
plot_calibration_curves(ytest_a, metrics_all['proba'], title='Model 2 Calibration Curves')
plt.show()

## Feature Importance Between Models

Gain importance for both models. Tick labels are colored by feature type
(statistical vs. complexity/temporal).

In [ ]:
feats_all = feature_matrix(X_all).columns    # 47 Model 2 features
feats_stat = feature_matrix(X_stat).columns  # 14 Model 1 features
plot_feature_importance_comparison(bst_all, feats_all, bst_stat, feats_stat, top_n=20)
plt.show()

## Principal Components Analysis (PCA) of Features

Features are z-scored on the training set; PCA is fit on train and applied to test.
The biplot colors each feature-type group and reports the angle between the two
group-mean loading vectors.

In [ ]:
Xtr_feats = feature_matrix(Xtrain_a)
Xte_feats = feature_matrix(Xtest_a)

scaler = StandardScaler()
pca = PCA(n_components=2)
pca.fit(scaler.fit_transform(Xtr_feats))
scores_test = pca.transform(scaler.transform(Xte_feats))

In [ ]:
ax, angle = pca_biplot(pca, scores_test, feature_names=list(Xte_feats.columns))
plt.show()
print(f'Angle between feature-type mean vectors: {angle} degrees')